In [21]:
# Import required libraries

import numpy as np
import pandas as pd
import pytest
from scipy.stats import ks_2samp
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

filepath = r"C:\Users\USER\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\86DED888A317FE2A325DB1FAFCECA89DDEB8FFD1\transfers\2026-47\Hotel_booking_project_cleaned with Dashboard_(2)(1).xlsx"
SHEET_NAME = "Cleaned_Data"
df = pd.read_excel(filepath, sheet_name="Cleaned_Data")


NUM_COLS = [
    "lead_time_days",
    "is_long_lead",
    "Stay Nights",
    "Guest Count",
    "log_amount_per_night",
    "log_total_amount",
    "Payment Duration (min)",
    "payment_share_of_booking",
    "is_weekend_checkin",
    "is_late_night_booking",
    "is_repeat_guest",
]
CAT_COLS = ["Room Type", "Payment Method"]
SEG_COLS = [
    "Stay Nights",
    "Guest Count",
    "log_amount_per_night",
    "log_total_amount",
    "lead_time_days",
]


def compute_psi(expected: np.ndarray, actual: np.ndarray, buckets: int = 5) -> float:
    """Population Stability Index (<0.10 no shift, 0.10-0.25 moderate, >0.25 major)."""
    quantiles = np.linspace(0, 100, buckets + 1)
    cut_points = np.percentile(expected, quantiles)
    cut_points[0] = -np.inf
    cut_points[-1] = np.inf

    exp_counts, _ = np.histogram(expected, bins=cut_points)
    act_counts, _ = np.histogram(actual, bins=cut_points)

    exp_pct = np.clip(exp_counts / max(len(expected), 1), 1e-6, None)
    act_pct = np.clip(act_counts / max(len(actual), 1), 1e-6, None)

    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


@pytest.fixture(scope="module")
def prepared_df() -> pd.DataFrame:
    df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
    df["lead_time_days"] = (
        df["Check-in Date"] - df["Booking Started Time"]
    ).dt.total_seconds() / 86400.0
    df["is_long_lead"] = (df["lead_time_days"] >= 30.0).astype(int)
    df["checkin_dayofweek"] = df["Check-in Date"].dt.dayofweek
    df["is_weekend_checkin"] = df["checkin_dayofweek"].isin([4, 5, 6]).astype(int)
    df["booking_start_hour"] = df["Booking Started Time"].dt.hour
    df["is_late_night_booking"] = (
        df["booking_start_hour"].between(22, 23) | df["booking_start_hour"].between(0, 5)
    ).astype(int)
    df["payment_share_of_booking"] = np.where(
        df["Booking Duration (min)"] > 0,
        df["Payment Duration (min)"] / df["Booking Duration (min)"],
        0.0,
    )
    df["log_amount_per_night"] = np.log1p(df["Amount Per Night"])
    df["log_total_amount"] = np.log1p(df["Total Amount"])
    df["user_total_bookings"] = df.groupby("User ID")["Booking ID"].transform("count")
    df["is_repeat_guest"] = (df["user_total_bookings"] > 1).astype(int)
    df["is_cancelled"] = (df["Booking Status"] == "Cancelled").astype(int)
    return df

def test_cancellation_model_evaluation_thresholds(prepared_df: pd.DataFrame):
    prep = ColumnTransformer([
        ("num", RobustScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])
    pipe = Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000, random_state=42)),
    ])

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
    scores = cross_validate(
        pipe,
        prepared_df[NUM_COLS + CAT_COLS],
        prepared_df["is_cancelled"],
        cv=cv,
        scoring={
            "pr_auc": "average_precision",
            "roc_auc": "roc_auc",
            "brier": "neg_brier_score",
        },
    )

    mean_pr_auc = float(np.mean(scores["test_pr_auc"]))
    mean_roc_auc = float(np.mean(scores["test_roc_auc"]))
    mean_brier = float(-np.mean(scores["test_brier"]))

    # Guard against accidental regression below baseline benchmarks
    assert mean_pr_auc >= 0.20, f"Unexpectedly low PR-AUC: {mean_pr_auc:.3f}"
    assert mean_roc_auc >= 0.50, f"Unexpectedly low ROC-AUC: {mean_roc_auc:.3f}"
    assert mean_brier <= 0.30, f"Unexpectedly high Brier loss: {mean_brier:.3f}"


def test_segmentation_silhouette_and_balance(prepared_df: pd.DataFrame):
    x_seg = StandardScaler().fit_transform(prepared_df[SEG_COLS])
    km = KMeans(n_clusters=3, n_init=30, random_state=42)
    labels = km.fit_predict(x_seg)

    sil = float(silhouette_score(x_seg, labels))
    assert sil >= 0.20, f"Low silhouette score: {sil:.3f}"

    cluster_sizes = pd.Series(labels).value_counts()
    assert (cluster_sizes >= 10).all(), f"Cluster too small: {cluster_sizes.to_dict()}"


def test_drift_detection_no_shift_on_split_and_detects_synthetic_shift(prepared_df: pd.DataFrame):
    half = len(prepared_df) // 2
    ref_lead = prepared_df["lead_time_days"].iloc[:half].to_numpy()
    cur_lead = prepared_df["lead_time_days"].iloc[half:].to_numpy()

    psi_same = compute_psi(ref_lead, cur_lead)
    ks_same = float(ks_2samp(ref_lead, cur_lead).pvalue)
    assert psi_same < 0.25
    assert ks_same > 0.05

   

In [22]:
# Guard against accidental regression below baseline benchmarks

def test_dataset_schema_and_invariants(prepared_df: pd.DataFrame):
    assert len(prepared_df) == 100
    assert prepared_df["User ID"].nunique() == 96
    assert prepared_df[NUM_COLS + CAT_COLS + ["is_cancelled"]].isna().sum().sum() == 0
    assert (prepared_df["lead_time_days"] > 0).all()
    assert prepared_df["payment_share_of_booking"].between(0.0, 1.0).all()

In [23]:
 # Simulate a severe distribution shift (e.g., sudden short-lead spike)

ref_lead = df["Stay Nights"].values    
shifted_lead = np.clip(ref_lead * 0.2, 0.1, None)
psi_shifted = compute_psi(ref_lead, shifted_lead)
ks_shifted = float(ks_2samp(ref_lead, shifted_lead).pvalue)
assert psi_shifted > 0.25
assert ks_shifted < 0.01